# Week 8 – PII Stress Test Assignment

## 1. Assignment Overview

### Objective

The objective of this activity is to test how effectively our existing PII (Personally Identifiable Information) scrubber detects and removes sensitive information from documents.

In the earlier Week 8 activities, we learned to use regular expressions and Presidio together for PII detection.

In this assignment, we deliberately provide a document containing different types of PII in unusual formats to identify where our existing scrubber succeeds and where it fails.

### What are we trying to achieve?

We will:

1. Prepare a document containing different types of PII.
2. Pass the document through our parsing, chunking and scrubbing pipeline.
3. Examine which PII patterns are detected and which are missed.
4. Document the failures and suggest improvements to the scrubber.

### Pipeline

Parse document → Chunk document → Scrub PII → Inspect results → Record findings

## 2. Prepare the Project Structure

Before executing the assignment, we prepared the required folders and Python modules inside the existing `demos` project directory.

### Working directory

`Demo Files/demos/`

### Folders and files created

| File / Folder | Purpose |
|---|---|
| `src/ingest/` | Stores the document ingestion modules. |
| `src/ingest/__init__.py` | Makes `ingest` a Python package. |
| `src/ingest/parsers.py` | Contains the `parse_document()` function to read the input document. |
| `src/ingest/chunkers.py` | Contains the `chunk_structurally()` function to divide the document into smaller chunks. |
| `src/ingest/pii.py` | Contains the `scrub_chunks()` function to detect and replace PII. |
| `data/stress_test/` | Stores the specially prepared PII stress-test document. |
| `docs/adr/` | Stores the findings and technical observations from the assignment. |

### Reusing the Week 8 code

We used the concepts and existing code from the Week 8 Day 1 and Day 2 notebooks and `wk08_pipeline.py`.

- Parsing logic was adapted to read a Markdown file.
- Structure-aware chunking was adapted to recognise Markdown headings.
- PII detection logic was adapted from the Week 8 Day 2 regex and Presidio implementation.

We created the required function interfaces so that the assignment's specified imports could be used without changing their names.

## 3. Prepare the Stress-Test Corpus

### What is a stress-test corpus?

A corpus is a collection of documents used for processing and analysis.

A stress-test corpus is deliberately designed to challenge a system by providing unusual, difficult or unexpected inputs.

For this assignment, we created a Markdown document containing different types of sensitive information in both normal and unusual formats.

### Input file

`data/stress_test/pii_adversarial.md`

### Types of information included

| Category | Examples |
|---|---|
| Names | Japanese, Arabic and Indian names |
| Email addresses | Obfuscated email formats and full-width characters |
| Phone numbers | International numbers, spelled-out digits and unusual spacing |
| Employee IDs | Different identifier formats |
| Addresses | Addresses from different countries |
| Financial information | IBAN and masked card number |

The document also contains a project budget code, which is not PII. This helps us examine whether the scrubber incorrectly identifies ordinary information as sensitive.

### Why are we using unusual formats?

A normal email address or phone number may be detected easily by a regular expression.

However, unusual spacing, spelling, special characters and unfamiliar identifier formats can cause detection failures.

The purpose of this document is to discover these weaknesses rather than assume that our scrubber will detect everything.

### Verification

We verified that:

- The corpus file exists in the expected location.
- The notebook is saved inside the `demos` directory.
- The notebook's working directory points to the correct project folder.

The corpus is now ready for processing.

In [1]:
from pathlib import Path

print("Current working directory:", Path.cwd())
print("Corpus file exists:", Path("data/stress_test/pii_adversarial.md").exists())

Current working directory: /voc/work/Demo Files/demos
Corpus file exists: True


### Step 4.1 – Import the required functions

Before executing the PII stress-test pipeline, we need to import the functions created in the `src/ingest` package.

- `parse_document`: Reads the Markdown document.
- `chunk_structurally`: Divides the parsed document into smaller chunks using its headings and text structure.
- `scrub_chunks`: Detects and replaces personally identifiable information (PII) in the chunks.

These functions are defined in three separate Python files: `parsers.py`, `chunkers.py`, and `pii.py`.

In [2]:
from src.ingest.parsers import parse_document
from src.ingest.chunkers import chunk_structurally
from src.ingest.pii import scrub_chunks

## 4.2 – Parse the Stress-Test Document

### What is parsing?

Parsing is the process of reading a document and extracting its contents into a format that Python can process.

In this activity, we use the `parse_document()` function defined in `src/ingest/parsers.py`.

The function:

1. Locates the input Markdown file.
2. Checks whether the file exists.
3. Reads its contents using UTF-8 encoding.
4. Returns a `ParsedDocument` object containing the extracted text and source filename.

The extracted text will be passed to the next stage of the pipeline: chunking.

In [3]:
# Parse
doc = parse_document("data/stress_test/pii_adversarial.md")

In [4]:
print("Source:", doc.source)
print("Total characters:", len(doc.text))
print("\nFirst 500 characters:\n")
print(doc.text[:500])

Source: pii_adversarial.md
Total characters: 1620

First 500 characters:

# Internal Memo — Project Khanjar

## Project team

Please direct queries to **Dr. Yuki Tanaka** (Tokyo office),
**Fatima Al-Rashid** (Dubai office), or **Rajesh Patel** (Mumbai
office). Aisha Mohammed will cover the European leg.

For US contacts: Carlos Mendoza is the regional lead — you can
reach him via the Atlanta office.

## Reaching out

You can email the team lead at alice [at] example [dot] com
(spaces matter — the form scrapes a lot of bots otherwise).

Alternative inbox is bob.smith ＠


## 4.3 – Chunk the Parsed Document

### What is chunking?

Chunking is the process of dividing a large document into smaller pieces called chunks.

Instead of processing the entire document as one large block of text, we divide it into manageable sections.

### Why do we need chunking?

- Smaller pieces are easier to process.
- Each chunk can be analysed independently.
- We can keep track of the document section to which each chunk belongs.
- PII detection and scrubbing can be performed on individual chunks.

### Structural chunking

In this assignment, we use `chunk_structurally()` from `src/ingest/chunkers.py`.

The function recognises Markdown headings and uses them to maintain the section information.

We specify `max_chars=800`, which sets the maximum target size of a chunk to 800 characters. Long text may be split further.

In [5]:
# Chunk
chunks = chunk_structurally(doc, max_chars=800)

In [6]:
print("Total number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i} ---")
    print("Section:", chunk.section_path)
    print("Characters:", len(chunk.text))
    print(chunk.text[:250])

Total number of chunks: 7

--- Chunk 0 ---
Section: Internal Memo — Project Khanjar > Project team
Characters: 279
Please direct queries to **Dr. Yuki Tanaka** (Tokyo office),

**Fatima Al-Rashid** (Dubai office), or **Rajesh Patel** (Mumbai

office). Aisha Mohammed will cover the European leg.

For US contacts: Carlos Mendoza is the regional lead — you can

reac

--- Chunk 1 ---
Section: Internal Memo — Project Khanjar > Reaching out
Characters: 296
You can email the team lead at alice [at] example [dot] com

(spaces matter — the form scrapes a lot of bots otherwise).

Alternative inbox is bob.smith ＠ example ．com (the at-sign and

dot are full-width Japanese characters, copy carefully).

Or use

--- Chunk 2 ---
Section: Internal Memo — Project Khanjar > Phone numbers
Characters: 247
- Yuki's office: +81 (0) 3 1234 5678

- Fatima's mobile: zero zero nine seven one five zero two three four five six seven

- Rajesh's desk: 91-22-2570-1234 ext. 4501

- Aisha's WhatsApp: +44 7700 900 123


### Chunking – Execution and Observation

The `chunk_structurally()` function successfully divided the parsed document into 7 chunks.

Each chunk contains:
- The extracted text belonging to that section.
- A `section_path` indicating its location in the original document.
- A character count showing the size of the chunk.

The output confirms that the document has been divided into smaller sections while retaining its structural information.

**Observation:** The chunking process is complete. These 7 chunks will now be passed to the PII detection and scrubbing stage.

## 4.4 – Scrub PII from the Chunks

### What is PII scrubbing?

PII scrubbing is the process of identifying personally identifiable information (PII) and replacing it with placeholders to prevent sensitive information from being exposed.

For example:

Original text:

`Contact Dr. Yuki Tanaka at alice@example.com`

Scrubbed text might look like:

`Contact [PERSON] at [EMAIL_ADDRESS]`

### Detection approaches used

Our `scrub_chunks()` function uses a hybrid approach:

**1. Regular Expressions (Regex)**

Regex searches text for predefined patterns.

Examples include:
- Email address patterns.
- Phone number patterns.
- Employee ID patterns.

**2. Presidio**

Presidio analyses text to identify sensitive entities using its detection engine.

It can identify entities such as person names and other recognised information types.

### Why combine both?

Regex is useful for predictable patterns, while Presidio can identify entities that do not always follow a fixed format.

Combining them provides broader detection coverage. However, neither approach guarantees that all PII will be detected.

This assignment is designed to discover those limitations.

In [8]:
# Scrub
scrubbed, scrub_report = scrub_chunks(chunks, use_presidio=True)

## 4.5 – Inspect the PII Scrubbing Results

### Why do we inspect the results?

Successful execution does not necessarily mean that all sensitive information has been detected.

The scrubber may:
- Detect and replace some PII correctly.
- Miss certain types of sensitive information.
- Detect information incorrectly.
- Partially redact sensitive information.

Therefore, we must inspect the output rather than simply assume that the scrubber worked perfectly.

### Information collected

For every chunk, we will examine:

- **Detected PII types:** The categories of sensitive information identified by the scrubber.
- **Number of redactions:** The number of replacements made in that chunk.
- **Scrubbed text:** The actual text after the detected PII has been replaced.

This inspection will help us identify the strengths and weaknesses of our existing scrubber.

In [9]:
# Inspect
for i, (chunk, report) in enumerate(zip(scrubbed, scrub_report)):
    print(f"\n--- Chunk {i} ---")
    print(f"Detected PII types: {report['detected_types']}")
    print(f"Redactions: {report['n_redactions']}")
    print(chunk.text[:400])
    print("...")


--- Chunk 0 ---
Detected PII types: ['LOCATION', 'NRP', 'PERSON']
Redactions: 11
Please direct queries to **Dr. <PERSON>** (<LOCATION> office),

**<PERSON>** (<LOCATION> office), or **<PERSON>** (<LOCATION>

office). <PERSON> will cover the <NRP> leg.

For <LOCATION> contacts: <PERSON> is the regional lead — you can

reach him via the <LOCATION> office.
...

--- Chunk 1 ---
Detected PII types: ['NRP', 'PERSON', 'URL']
Redactions: 3
You can email the team lead at <PERSON> [at] example [dot] com

(spaces matter — the form scrapes a lot of bots otherwise).

Alternative inbox is <URL>ith ＠ example ．com (the at-sign and

dot are full-width <NRP> characters, copy carefully).

Or use the secure form: y.tanaka(at)hospital(dot)org
...

--- Chunk 2 ---
Detected PII types: ['DATE_TIME', 'PERSON', 'PHONE', 'PHONE_NUMBER']
Redactions: 6
- Yuki's office: +81 (0) 3 [PHONE]

- <PERSON>'s mobile: zero zero nine seven one five zero two three four five six seven

- Rajesh's desk: 91-22-[PHONE] ext. 4501

### PII Scrubbing – First Run Observations

The hybrid scrubber successfully processed all 7 chunks and reported a total of 41 redactions.

However, the results revealed several limitations:

1. **Names:** Most team member names were detected and replaced.
2. **Obfuscated emails:** The three unusual email formats were not successfully removed.
3. **Phone numbers:** Several phone numbers were only partially redacted. The spelled-out phone number was missed.
4. **Employee IDs:** The standard `EMP-` format was detected, but other custom identifier formats were missed.
5. **Addresses:** Some address components were detected, but complete address removal was inconsistent.
6. **Financial information:** The IBAN was partially identified as a phone number, while the masked card number was missed.
7. **Incorrect classifications:** Some information was assigned inappropriate entity types, such as an employee badge being classified as a US driver's licence.

The results demonstrate that successful execution does not guarantee complete or accurate PII removal.

**Conclusion:** The first run establishes a baseline for evaluating the existing scrubber. We must compare the original document with the scrubbed output item by item to determine exactly what was caught, missed or partially redacted.

## 5 – Build the PII Failure Log

### 5.1 – Create the Findings File

After running the scrubber, we need to document which PII patterns were detected, missed or only partially redacted.

We will create a Markdown file named `pii_stress_test_findings.md` inside `docs/adr/`.

This file will serve as a record of the first-run results.

It will contain:
- A summary of the test results.
- A category-wise comparison of the original PII items and the scrubbed output.
- Specific observations about missed and partially redacted information.
- Proposed improvements to the existing scrubber.

The findings log is important because it helps us move from simply observing that the scrubber failed to identifying exactly what needs improvement.

## Step 6 – Proposed Improvements

Based on the first-run results, the following improvements are proposed.

### Fix 1 – Employee ID detection

**Problem:**

The existing regex detects the EMP- format but misses other employee identifiers such as Staff #4082, RJP/2019/0451 and A.M.-2021-EUR-007.

**Proposed fix:**

Add custom regular expressions for the additional employee ID formats.

**Example code:**

```python
EMPLOYEE_ID_PATTERNS = [
    r"\bEMP-\d{4,6}\b",
    r"\bStaff\s*#\s*\d{3,6}\b",
    r"\b[A-Z]{2,3}/\d{4}/\d{3,5}\b",
    r"\b[A-Z]\.[A-Z]\.-\d{4}-[A-Z]{2,4}-\d{3}\b",
]

Trade-off:

Custom patterns improve detection of known ID formats, but new formats may require additional rules. Broad patterns may also produce false positives.
Fix 2 – Obfuscated email detection
Problem:

The scrubber misses email addresses written with alternatives such as [at], [dot], full-width characters and the words (at) and (dot).

Proposed fix:

Normalise common obfuscated email formats before applying email detection.

For example:
- Replace [at] and (at) with @.
- Replace [dot] and (dot) with ..
- Convert full-width ＠ to @.
- Convert full-width ． to ..

In [10]:
import unicodedata

def normalise_obfuscated_email(text):
    text = unicodedata.normalize("NFKC", text)

    text = re.sub(
        r"\s*\[(?:at)\]\s*|\s*\((?:at)\)\s*",
        "@",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s*\[(?:dot)\]\s*|\s*\((?:dot)\)\s*",
        ".",
        text,
        flags=re.IGNORECASE
    )

    return text

Trade-off:

Normalisation helps identify deliberately altered email addresses, but replacing words such as "at" and "dot" without sufficient context could alter ordinary text. The normalisation rules should be applied carefully.
Fix 3 – Phone number detection
Problem:

Some international phone numbers are only partially redacted, while spelled-out phone numbers are missed entirely.

Proposed fix:

Use a dedicated phone-number detection library or improved country-aware phone patterns. Validate complete phone-number matches instead of replacing only a portion.

A library such as phonenumbers can help parse and validate international phone number formats.

In [11]:
import phonenumbers

def detect_phone_numbers(text):
    matches = []

    for match in phonenumbers.PhoneNumberMatcher(
        text,
        None,
        leniency=phonenumbers.Leniency.POSSIBLE
    ):
        matches.append({
            "start": match.start,
            "end": match.end,
            "number": match.raw_string
        })

    return matches

import phonenumbersdef detect_phone_numbers(text):    matches = []    for match in phonenumbers.PhoneNumberMatcher(        text,        None,        leniency=phonenumbers.Leniency.POSSIBLE    ):        matches.append({            "start": match.start,            "end": match.end,            "number": match.raw_string        })    return matches



Trade-off:

A dedicated library can improve international number detection, but it may not recognise numbers written entirely as words. Additional normalisation is needed for spelled-out digits.
Fix 4 – Address and financial information detection
Problem:

The scrubber partially redacts addresses and does not reliably identify IBANs or masked card numbers.

Proposed fix:

Introduce dedicated detection rules for financial identifiers and improve address handling.

For financial information, use specific patterns for IBANs and masked card numbers. For addresses, consider combining Presidio location detection with contextual address patterns.

In [16]:
import re
FINANCIAL_PATTERNS = {
    "IBAN": re.compile(
        r"\b[A-Z]{2}\d{2}(?:[ \t-]*[A-Z0-9]){11,30}\b",
        re.IGNORECASE
    ),
    "MASKED_CARD": re.compile(
        r"\*{4}-\*{4}-\*{4}-\d{4}"
    )
}

Trade-off:

Specific patterns can improve financial information detection, but unusual formats may still be missed. Address detection is more context-dependent and can accidentally redact ordinary location references.
Overall recommendation
The scrubber should use multiple layers:
1. Text normalisation for unusual formats.
2. Custom regex patterns for known identifiers.
3. Presidio for named entities and contextual detection.
4. Manual auditing to identify information that automated methods miss.
No single detection method can guarantee complete PII protection.

# Step 3 – Test the patterns

In [18]:
test_text = """
IBAN: GB82 WEST 1234 5698 7654 32
Card: ****-****-****-4521
"""

for label, pattern in FINANCIAL_PATTERNS.items():
    matches = pattern.findall(test_text)
    print(label, ":", matches)

IBAN : ['GB82 WEST 1234 5698 7654 32']
MASKED_CARD : ['****-****-****-4521']


We have completed the independent test of the financial-data patterns. Now let's review the proposed fixes and make sure the findings document explains what each fix addresses, how it works, and what its limitations are.
Step 6. Review the proposed fixes
The stress test showed that the original scrubber does not recognise every PII format. We proposed four improvements.
Fix 1: Improve employee ID detection
Problem observed: The original regex recognises EMP-04827, but does not reliably recognise other formats such as:
- Staff #4082
- RJP/2019/0451
- A.M.-2021-EUR-007
Proposed fix: Add separate patterns for the different ID formats.

In [19]:
EMPLOYEE_ID_PATTERNS = [
    re.compile(r"\bEMP-\d{4,6}\b", re.IGNORECASE),
    re.compile(r"\bStaff\s*#\s*\d{3,6}\b", re.IGNORECASE),
    re.compile(r"\b[A-Z]{2,3}/\d{4}/\d{3,5}\b"),
]

Trade-off: These patterns can detect the specified formats, but employee ID formats vary between organisations. A pattern may also match something that is not actually an employee ID.

Note: The A.M.-2021-EUR-007 format needs its own carefully designed pattern; it is not covered by the three patterns above.
Fix 2: Normalise obfuscated email addresses
Problem observed: The original email regex does not recognise formats such as:
- alice [at] example [dot] com
- y.tanaka(at)hospital(dot)org
Proposed fix: Convert common obfuscated forms into normal email syntax before applying the email regex.

In [20]:
import unicodedata
import re

def normalise_obfuscated_email(text):
    text = unicodedata.normalize("NFKC", text)

    text = re.sub(
        r"\s*\[(?:at)\]\s*|\s*\((?:at)\)\s*",
        "@",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s*\[(?:dot)\]\s*|\s*\((?:dot)\)\s*",
        ".",
        text,
        flags=re.IGNORECASE
    )

    return text

Trade-off: Normalisation helps with the specific formats covered by these rules. It does not handle every possible obfuscation, and it should be followed by email detection and redaction.
Fix 3: Use a phone-number library
Problem observed: Phone numbers can contain country codes, spaces, brackets, extensions, or spelled-out digits. A simple regex may capture only part of a number or miss it entirely.

Proposed fix: Use the phonenumbers Python library.

In [21]:
import phonenumbers

def detect_phone_numbers(text):
    matches = []

    for match in phonenumbers.PhoneNumberMatcher(
        text,
        None,
        leniency=phonenumbers.Leniency.POSSIBLE
    ):
        matches.append({
            "start": match.start,
            "end": match.end,
            "number": match.raw_string
        })

    return matches

Trade-off: The library is designed for conventional phone-number formats. It will not automatically understand every spelled-out or deliberately distorted number. Passing None as the region may also limit recognition of numbers without country codes.
Fix 4: Improve financial-data detection
Problem observed: The original scrubber partially detects the IBAN as a phone number and does not redact the masked card number.

Proposed fix: Add dedicated patterns for financial information.

We independently tested these patterns, and both produced the expected output:

In [22]:
FINANCIAL_PATTERNS = {
    "IBAN": re.compile(
        r"\b[A-Z]{2}\d{2}(?:[ \t-]*[A-Z0-9]){11,30}\b",
        re.IGNORECASE
    ),
    "MASKED_CARD": re.compile(
        r"\*{4}-\*{4}-\*{4}-\d{4}"
    )
}

Trade-off: These patterns cover the formats in this assignment, not every possible IBAN or card format. They need broader testing before being used in a real system.

Step 6.1. Compare the proposed fixes
Fix	Main problem addressed	Current status
Employee ID patterns	Different employee ID formats	Proposed, not tested
Email normalisation	Obfuscated email addresses	Proposed, not tested
Phone-number library	International phone formats	Proposed, not tested
Financial patterns	IBAN and masked card	Independently tested

Important distinction: Only the financial patterns have been tested independently. None of these four fixes has yet been integrated into the original scrub_chunks() function.

Step 6.2. Add the review to your findings document
Open:

docs/adr/pii_stress_test_findings.md

At the end of the file, add this heading and table:

Let's implement the **financial-data detection fix** as our optional implementation activity.

We will make one focused change to `src/ingest/pii.py`, rerun the pipeline, and compare the new results with the original baseline.

## **Step 7. Implement the financial-data fix**

### **7.1 Understand what we are changing**

Currently, your `PII_PATTERNS` dictionary contains three patterns:

- `EMAIL`
- `PHONE`
- `EMP_ID`

We will add two more:

- `IBAN` – to recognise international bank account numbers.
- `MASKED_CARD` – to recognise masked card numbers.

One important detail: we will place these two patterns **before the PHONE pattern**. Otherwise, the phone regex might partially match the digits inside an IBAN before the IBAN pattern gets a chance to detect it.

### **7.2 Open the Python file**

Open this file in VS Code:

`src/ingest/pii.py`

Find the existing section beginning with:

```
PII_PATTERNS = {
```

Replace the entire existing `PII_PATTERNS` dictionary with this:

```
PII_PATTERNS = {    "EMAIL": re.compile(        r'\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b'    ),    # Added for the Week 8 PII stress test    "IBAN": re.compile(        r"\b[A-Z]{2}\d{2}(?:[ \t-]*[A-Z0-9]){11,30}\b",        re.IGNORECASE    ),    "MASKED_CARD": re.compile(        r"\*{4}-\*{4}-\*{4}-\d{4}"    ),    "PHONE": re.compile(        r'\+?\d{0,2}[-.\s]?\(?\d{3}\)?[-.\s]?\d{3,4}(?:[-.\s]?\d{4})?\b'    ),    "EMP_ID": re.compile(r'\bEMP-\d{4,6}\b'),}
```

**Do not change any other part of&#x20;**`pii.py`**&#x20;yet.** In particular, leave the `scrub_chunks()` function unchanged.

### **7.3 Save the file**

Press `Ctrl + S`.

Expected change

Your original three patterns are retained, and two new patterns have been added.

**Stop here and tell me once you have saved&#x20;**`src/ingest/pii.py`**.** Then we will reload the updated code in the notebook and rerun the scrubber without losing the original baseline results.

# Step 7.4. Reload the updated Python file
### Reload the Updated PII Module

We have added two new detection patterns (IBAN and MASKED_CARD) to `src/ingest/pii.py`.

The notebook may still be using the previously imported version of this file. We use Python's `importlib.reload()` function to reload the module and make the updated code available.

We keep the original baseline results in separate variables so that we can compare them with the updated results later.

In [23]:
import importlib
import src.ingest.pii

importlib.reload(src.ingest.pii)

print("Updated PII module loaded successfully.")

Updated PII module loaded successfully.


# Step 7.5. Run the updated scrubber
The updated PII module has loaded successfully. Now let's run the modified scrubber.

We will store the new results in different variables so that the original baseline results remain available for comparison.

### Run the Updated PII Scrubber

We have added IBAN and MASKED_CARD detection patterns to the PII module.

We now run the same stress-test chunks through the updated scrubber.

The updated results are stored in `scrubbed_updated` and `scrub_report_updated`. This allows us to compare the new results with the original baseline results without overwriting them.

In [24]:
scrubbed_updated, scrub_report_updated = src.ingest.pii.scrub_chunks(
    chunks,
    use_presidio=True
)

print("Updated PII scrubbing completed successfully.")

Updated PII scrubbing completed successfully.


### Step 7.6 Inspect the Updated Financial-Data Detection
Remember, the financial information appears in Chunk 5 of our original corpus.
We added two new patterns to detect IBANs and masked card numbers.

We now inspect Chunk 5, which contains the financial information in our stress-test document.

We compare the original baseline output with the updated output to check whether the new patterns have detected and redacted these values correctly.

In [25]:
print("===== ORIGINAL BASELINE =====")
print("Detected PII types:", scrub_report[5]["detected_types"])
print("Redactions:", scrub_report[5]["n_redactions"])
print(scrubbed[5].text)

print("\n===== UPDATED SCRUBBER =====")
print("Detected PII types:", scrub_report_updated[5]["detected_types"])
print("Redactions:", scrub_report_updated[5]["n_redactions"])
print(scrubbed_updated[5].text)

===== ORIGINAL BASELINE =====
Detected PII types: ['PHONE']
Redactions: 1
- Project budget code: PRJ-2024-KHANJAR-EU

- Lead's IBAN for expenses: GB82 WEST [PHONE] 32

- Petty cash card last four: ****-****-****-4521

===== UPDATED SCRUBBER =====
Detected PII types: ['IBAN', 'MASKED_CARD']
Redactions: 2
- Project budget code: PRJ-2024-KHANJAR-EU

- Lead's IBAN for expenses: [IBAN]

- Petty cash card last four: [MASKED_CARD]


Excellent. The comparison confirms that our financial-data fix is working correctly.

Fix successfully implemented and tested

## **Step 7.7. Understand the before-and-after results**

| Financial information | Original baseline             | Updated scrubber                  |
| --------------------- | ----------------------------- | --------------------------------- |
| IBAN                  | Partially detected as `PHONE` | Fully redacted as `[IBAN]`        |
| Masked card           | Not detected                  | Fully redacted as `[MASKED_CARD]` |
| Project code          | Retained                      | Retained                          |

### **What changed?**

**1. IBAN detection**

Original:

```
Lead's IBAN for expenses: GB82 WEST [PHONE] 32
```

Updated:

```
Lead's IBAN for expenses: [IBAN]
```

The original phone-number regex incorrectly identified part of the IBAN as a phone number. Our new IBAN pattern recognises the complete value.

**2. Masked card detection**

Original:

```
Petty cash card last four: ****-****-****-4521
```

Updated:

```
Petty cash card last four: [MASKED_CARD]
```

The new pattern successfully detects the masked card format.

**3. Project code**

```
Project budget code: PRJ-2024-KHANJAR-EU
```

It remains unchanged. This is correct because the assignment identifies this project code as non-PII.

### **Redaction count comparison**

Original baseline

# **1**

Redaction in Chunk 5

Updated scrubber

# **2**

Redactions in Chunk 5

The updated scrubber performs two correct financial-data redactions instead of one incorrect partial detection.

## **Step 7.8. Document the implementation**

Open `docs/adr/pii_stress_test_findings.md` and append:

```
## Optional Implementation – Financial Data Detection

### Changes Made

Two new regular expressions were added to the PII_PATTERNS
dictionary in src/ingest/pii.py:

1. IBAN – detects international bank account numbers.
2. MASKED_CARD – detects masked card-number formats.

The original EMAIL, PHONE and EMP_ID patterns were retained.

### Before-and-After Results

| Financial Information | Original Baseline | Updated Scrubber |
|---|---|---|
| IBAN | Partially detected as PHONE | Fully detected as IBAN |
| Masked card | Not detected | Fully detected as MASKED_CARD |
| Project code | Correctly retained | Correctly retained |

### Conclusion

The implementation successfully improved financial-data
detection in Chunk 5.

The redaction count increased from 1 to 2.

The project code remained unchanged because it is not PII.

The original baseline results were preserved in separate
notebook variables for comparison.

### Limitation

The new patterns were tested against the financial formats
in this assignment. They have not been validated against
all possible IBAN and masked-card formats.
```

## **Progress tracker**

WEEK 8 PII STRESS TEST

Corpus creation and parsing

Structural chunking

Baseline PII scrubbing

Baseline findings document

Propose four improvements

Implement financial-data fix

Compare before and after

Update findings document

Update wk8-snapshot.md

**Next:** Save the findings document. Then we will update `wk8-snapshot.md` to record the activity and its outcome, completing the assignment documentation.